# Run the BI Mart Workflow

This notebook calls the existing pipeline modules to prepare Silver Parquet files, build the mart, and load it into SQL Server. No production transformation logic is duplicated in the notebook.

## Before You Run

- Use Python 3.12 and install dependencies with `uv sync --locked`.
- Select the `.venv` Python environment kernel in VS Code/Jupyter.
- Download the Kaggle dataset separately; CSV and Parquet files are not stored in Git. Place the files as specified in `configs/data_contract.yml`.
- To create Silver files from CSV, set `PREPARE_SILVER_FROM_RAW = True` in the configuration cell and run the notebook from the beginning.
- SQL Server must be running. Configure the connection in `.env`; the notebook does not display the connection string or password.
- `PUBLISH_TO_SQL_SERVER` is disabled by default. When enabled, the loader replaces the existing mart tables in the configured database.

To use a local Docker SQL Server, configure `.env` according to `.env.example` and start the SQL Server container separately. An externally installed SQL Server can use Windows Authentication with `MSSQL_TRUSTED_CONNECTION=true`; the Windows account must have database permissions.

In [ ]:
import logging
import os
import sys
from pathlib import Path

from IPython.display import display

SUPPORTED_PYTHON_VERSION = (3, 12)
SQL_CONNECTION_TIMEOUT_SECONDS = 8

workspace_start = Path.cwd().resolve()
repository_candidates = (workspace_start, *workspace_start.parents)
repository_matches = [
    candidate
    for candidate in repository_candidates
    if (candidate / "pyproject.toml").is_file()
]
if not repository_matches:
    raise FileNotFoundError("Could not find the repository root containing pyproject.toml")

PROJECT_ROOT = repository_matches[0]
os.chdir(PROJECT_ROOT)
source_directory = str(PROJECT_ROOT / "src")
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

if sys.version_info[:2] != SUPPORTED_PYTHON_VERSION:
    raise RuntimeError(
        f"Select the project Python {SUPPORTED_PYTHON_VERSION[0]}.{SUPPORTED_PYTHON_VERSION[1]} kernel"
    )

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
display(
    {
        "repository": str(PROJECT_ROOT),
        "python": sys.version.split()[0],
        "kernel_ready": True,
    }
)

## Configure the Run

Keep `PREPARE_SILVER_FROM_RAW = False` if all seven Silver Parquet files already exist. Set it to `True` to have the notebook run ingestion, contract validation, application normalization, normalization of the six history tables, and history-flag generation in sequence.

Set `PUBLISH_TO_SQL_SERVER = True` only after confirming that the host/database in `.env` is the intended target. Publishing replaces the existing mart tables with the staging data in a transaction.

In [ ]:
PREPARE_SILVER_FROM_RAW = False
PUBLISH_TO_SQL_SERVER = False

## Optional: Create Bronze and Silver from CSV

This cell does nothing when `PREPARE_SILVER_FROM_RAW` is disabled. When enabled, missing CSV files or an invalid contract will stop the cell with a specific error.

In [ ]:
if PREPARE_SILVER_FROM_RAW:
    from hcr.contracts.validate_raw import run as validate_raw
    from hcr.ingest.to_parquet import main as ingest_raw
    from hcr.silver.clean_application import main as clean_application
    from hcr.silver.clean_history import run as clean_history
    from hcr.silver.flags import build_history_flags

    ingest_raw()
    contract_reports = validate_raw()
    clean_application()
    history_row_counts = clean_history()
    history_flag_rows = build_history_flags()
    preparation_result = {
        "contract_tables_validated": len(contract_reports),
        "history_tables_cleaned": history_row_counts,
        "history_flag_rows": history_flag_rows,
    }
else:
    preparation_result = "Skipped raw-to-Silver preparation; using existing Silver Parquet files."

display(preparation_result)

## Check Silver Inputs

In [ ]:
from hcr.paths import silver_path
from hcr.serving.build_star import SOURCE_TABLES

silver_paths = {table: silver_path(table) for table in SOURCE_TABLES}
missing_silver_paths = [path for path in silver_paths.values() if not path.is_file()]
if missing_silver_paths:
    missing_list = "\n".join(str(path) for path in missing_silver_paths)
    raise FileNotFoundError(
        "Required Silver Parquet files are missing. Run the preparation cell or restore them:\n"
        + missing_list
    )

silver_status = [
    {"table": table, "path": str(path), "size_bytes": path.stat().st_size}
    for table, path in silver_paths.items()
]
display(silver_status)

## Build the Parquet Mart

Build the fact and dimension tables with DuckDB from the seven Silver Parquet files. This step writes only to `data/mart/`; it does not replace any SQL Server tables.

In [ ]:
from hcr.serving.build_star import run as build_serving_mart

mart_row_counts = build_serving_mart()
display(mart_row_counts)

## Check the SQL Server Connection

This cell only opens a connection and reads the current database/login. It does not expose the connection string, SQL username, or password.

In [ ]:
import pyodbc

from hcr.config import mssql_connection_string, mssql_database_name

database_name = mssql_database_name()
connection = pyodbc.connect(
    mssql_connection_string(),
    timeout=SQL_CONNECTION_TIMEOUT_SECONDS,
)
try:
    connection_row = connection.cursor().execute(
        "SELECT DB_NAME(), SUSER_SNAME()"
    ).fetchone()
    if connection_row is None:
        raise RuntimeError("SQL Server returned no connection identity")
    if str(connection_row[0]).casefold() != database_name.casefold():
        raise RuntimeError(
            f"Connected to database {connection_row[0]}, expected {database_name}"
        )
    sql_connection_status = {
        "database": str(connection_row[0]),
        "login": str(connection_row[1]),
        "connected": True,
    }
finally:
    connection.close()

display(sql_connection_status)

## Load and Verify the Mart in SQL Server

Publishing is skipped by default. After verifying the target database, set `PUBLISH_TO_SQL_SERVER` to `True` in the configuration cell, then run this cell again. The loader creates staging tables, replaces the mart tables in a transaction, and validates the foreign keys.

In [ ]:
if PUBLISH_TO_SQL_SERVER:
    from hcr.serving.build_star import EXPECTED_APPLICATION_ROWS, MART_TABLES
    from hcr.serving.load_sqlserver import run as load_serving_mart

    load_report = load_serving_mart()
    verification_connection = pyodbc.connect(
        mssql_connection_string(),
        timeout=SQL_CONNECTION_TIMEOUT_SECONDS,
    )
    try:
        cursor = verification_connection.cursor()
        loaded_row_counts = {
            table: int(
                cursor.execute(f"SELECT COUNT_BIG(*) FROM dbo.[{table}]").fetchone()[0]
            )
            for table in MART_TABLES
        }
        primary_key_count = int(
            cursor.execute(
                "SELECT COUNT(*) FROM sys.key_constraints WHERE type = 'PK'"
            ).fetchone()[0]
        )
        foreign_key_count = int(
            cursor.execute(
                "SELECT COUNT(*) FROM sys.foreign_keys WHERE is_disabled = 0"
            ).fetchone()[0]
        )
        remaining_staging_tables = int(
            cursor.execute(
                "SELECT COUNT(*) FROM sys.tables WHERE name LIKE 'stg[_]%'"
            ).fetchone()[0]
        )
    finally:
        verification_connection.close()

    row_count_mismatches = {
        table: {"expected": mart_row_counts[table], "actual": loaded_row_counts[table]}
        for table in MART_TABLES
        if loaded_row_counts[table] != mart_row_counts[table]
    }
    if row_count_mismatches:
        raise RuntimeError(f"SQL Server row counts differ from the mart: {row_count_mismatches}")
    if loaded_row_counts["Fact_Application"] != EXPECTED_APPLICATION_ROWS:
        raise RuntimeError("Fact_Application row count does not match the data contract")
    if load_report.orphan_keys != 0:
        raise RuntimeError(f"SQL Server reported {load_report.orphan_keys} orphan foreign keys")
    if remaining_staging_tables != 0:
        raise RuntimeError(f"Found {remaining_staging_tables} staging tables after publish")

    publish_result = {
        "database": database_name,
        "table_row_counts": loaded_row_counts,
        "primary_keys": primary_key_count,
        "foreign_keys": foreign_key_count,
        "orphan_keys": load_report.orphan_keys,
        "remaining_staging_tables": remaining_staging_tables,
    }
else:
    publish_result = "Skipped SQL publish. Set PUBLISH_TO_SQL_SERVER = True to replace serving tables."

display(publish_result)